<a href="https://colab.research.google.com/github/thienbao21022k7-droid/Cuoiky/blob/main/file_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+http://github.com/facebookresearch/segment-anything.git
!pip install opencv-python matplotlib

  Cloning http://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-8hsmf_7t
  Running command git clone --filter=blob:none --quiet http://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-8hsmf_7t
  Resolved http://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done


In [ ]:
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

--2026-06-21 01:51:03--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.4, 65.9.168.62, 65.9.168.52, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.4|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2564550879 (2.4G) [binary/octet-stream]
Saving to: ‘sam_vit_h_4b8939.pth.1’

sam_vit_h_4b8939.pt 100%[===================>]   2.39G   120MB/s    in 18s     

2026-06-21 01:51:21 (140 MB/s) - ‘sam_vit_h_4b8939.pth.1’ saved [2564550879/2564550879]



In [ ]:
!pip install streamlit pandas -q

In [ ]:
%%writefile app.py
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
import streamlit as st
import hashlib
import os
from torchvision import models, transforms
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# ==========================================
# CẤU HÌNH GIAO DIỆN STREAMLIT
# ==========================================
st.set_page_config(page_title="AI Tiệm Bánh", page_icon="🍰", layout="wide")

# ==========================================
# HÀM BỔ TRỢ: XÓA DẤU TIẾNG VIỆT
# ==========================================
def remove_accents(input_str):
    s1 = u'ÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝàáâãèéêìíòóôõùúýĂăĐđĨĩŨũƠơƯưẠạẢảẤấẦầẨẩẪẫẬậẮắẰằẲẳẴẵẶặẸẹẺẻẼẽẾếỀềỂểỄễỆệỈỉỊịỌọỎỏỐốỒồỔổỖỗỘộỚớỜờỞởỠỡỢợỤụỦủỨứỪừỬửỮữỰựỲỳỴỵỶỷỸỹ'
    s0 = u'AAAAEEEIIOOOOUUYaaaaeeeiioooouuyAaDdIiUuOoUuAaAaAaAaAaAaAaAaAaAaAaAaAaEeEeEeEeEeEeEeEeIiIiOoOoOoOoOoOoOoOoOoOoOoOoUuUuUuUuUuUuUuYyYyYyYy'
    s = ''
    for c in input_str:
        if c in s1:
            s += s0[s1.index(c)]
        else:
            s += c
    return s

# ==========================================
# 1. KHỞI TẠO BỘ NHỚ VÀ ĐƯỜNG DẪN
# ==========================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
sam_checkpoint = "sam_vit_h_4b8939.pth"
cnn_model_path = '/content/drive/MyDrive/cake_classifier.pth'
qr_code_path = '/content/qr.png'

class_names = [
    'Bánh chuối nướng', 'Bánh da lợn', 'Bánh mì bơ (Cua lớn)',
    'Bánh mì dừa lưới', 'Chà bông cây', 'Cookies dừa',
    'Croissant', 'Egg Tart', 'Muffin Việt Quất', 'Patechaud'
]
num_classes = len(class_names)

if 'menu' not in st.session_state:
    st.session_state.menu = pd.DataFrame({
        "Tên Bánh (AI Nhận Diện)": class_names,
        "Giá Tiền (VNĐ)": [19000, 23000, 18000, 15000, 27000, 23000, 30000, 21000, 25000, 30000]
    })

if 'detections' not in st.session_state:
    st.session_state.detections = []
if 'analyzed_image' not in st.session_state:
    st.session_state.analyzed_image = None
if 'last_file_hash' not in st.session_state:
    st.session_state.last_file_hash = ""

# ==========================================
# 2. HÀM TẢI MÔ HÌNH
# ==========================================
@st.cache_resource
def load_models():
    sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint)
    sam.to(device=device)
    mask_gen = SamAutomaticMaskGenerator(
        model=sam, points_per_side=32, pred_iou_thresh=0.88,
        stability_score_thresh=0.95, crop_n_layers=1,
        crop_n_points_downscale_factor=2, min_mask_region_area=5000
    )

    cnn = models.resnet18(weights=None)
    cnn.fc = torch.nn.Linear(cnn.fc.in_features, num_classes)
    cnn.load_state_dict(torch.load(cnn_model_path, map_location=device))
    cnn = cnn.to(device)
    cnn.eval()

    trans = transforms.Compose([
        transforms.Resize((224, 224)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return mask_gen, cnn, trans

# ==========================================
# 3. GIAO DIỆN CHÍNH
# ==========================================
st.title("🍰 Hệ Thống AI Tiệm Bánh Tự Động")

with st.spinner("Đang khởi động AI..."):
    try:
        mask_generator, cnn_model, cnn_transforms = load_models()
    except Exception as e:
        st.error(f"Lỗi tải mô hình. Chi tiết: {e}")
        st.stop()

tab1, tab2 = st.tabs(["📸 Nhận Diện & Lên Đơn", "⚙️ Bảng Quản Lý Giá"])

with tab2:
    st.header("⚙️ Chỉnh Sửa Giá Bánh")
    edited_df = st.data_editor(
        st.session_state.menu,
        use_container_width=True, hide_index=True,
        column_config={
            "Tên Bánh (AI Nhận Diện)": st.column_config.TextColumn(disabled=True),
            "Giá Tiền (VNĐ)": st.column_config.NumberColumn("Giá Tiền (VNĐ)", min_value=0, step=1000, required=True, format="%d")
        }
    )
    st.session_state.menu = edited_df

with tab1:
    st.header("📸 Tự Động Lên Đơn Hàng")
    uploaded_file = st.file_uploader("Tải lên ảnh khay bánh của khách", type=["jpg", "jpeg", "png"])

    if uploaded_file is not None:
        file_bytes = uploaded_file.getvalue()
        file_hash = hashlib.md5(file_bytes).hexdigest()
        if file_hash != st.session_state.last_file_hash:
            st.session_state.last_file_hash = file_hash
            st.session_state.detections = []
            st.session_state.analyzed_image = None

        image = Image.open(uploaded_file).convert('RGB')
        image_np = np.array(image)

        if st.button("🔍 Xử lý ảnh bằng AI", type="primary"):
            with st.spinner("AI đang lọc vụn thừa, phân tích và tra cứu giá..."):
                st.session_state.analyzed_image = image_np.copy()
                st.session_state.detections = []

                masks = mask_generator.generate(image_np)
                masks = sorted(masks, key=lambda x: x['area'], reverse=True)
                drawn_boxes = []

                for mask in masks:
                    x, y, w, h = mask['bbox']
                    if w > image_np.shape[1]*0.8 and h > image_np.shape[0]*0.8: continue
                    if w < 100 or h < 100: continue

                    box1 = [x, y, x+w, y+h]
                    box1_area = w * h
                    is_overlap = False
                    for box2 in drawn_boxes:
                        ix1, iy1 = max(box1[0], box2[0]), max(box1[1], box2[1])
                        ix2, iy2 = min(box1[2], box2[2]), min(box1[3], box2[3])
                        iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
                        if iw > 0 and ih > 0:
                            if (iw * ih) / box1_area > 0.4:
                                is_overlap = True
                                break
                    if is_overlap: continue
                    drawn_boxes.append(box1)

                    cropped_cake_np = image_np[y:y+h, x:x+w]
                    pil_img = Image.fromarray(cropped_cake_np)
                    input_tensor = cnn_transforms(pil_img).unsqueeze(0).to(device)

                    with torch.no_grad():
                        outputs = cnn_model(input_tensor)
                        _, predicted_idx = torch.max(outputs, 1)
                        cake_name = class_names[predicted_idx.item()]

                    # Bổ sung trường 'quantity' mặc định là 1
                    st.session_state.detections.append({
                        'bbox': box1,
                        'crop': pil_img,
                        'corrected_class': cake_name,
                        'quantity': 1
                    })

        # ==========================================
        # KHU VỰC CHỈNH SỬA & TỔNG HỢP HÓA ĐƠN
        # ==========================================
        if len(st.session_state.detections) > 0:
            st.success("✅ Hoàn tất nhận diện! Bạn có thể sửa TÊN và SỐ LƯỢNG ở bên dưới. Đưa số lượng về 0 để xóa bánh lỗi.")

            st.markdown("### ✍️ Kiểm tra & Sửa đổi món")
            cols = st.columns(4)
            for i, det in enumerate(st.session_state.detections):
                col = cols[i % 4]
                with col:
                    st.image(det['crop'], use_container_width=True)
                    # Chọn tên
                    new_class = st.selectbox(
                        f"Tên món #{i+1}",
                        options=class_names,
                        index=class_names.index(det['corrected_class']),
                        key=f"edit_cake_{i}"
                    )
                    # Chọn số lượng
                    new_qty = st.number_input(
                        f"Số lượng #{i+1}",
                        min_value=0, # Cho phép chỉnh về 0 để xóa
                        value=det.get('quantity', 1),
                        step=1,
                        key=f"edit_qty_{i}"
                    )

                    det['corrected_class'] = new_class
                    det['quantity'] = new_qty

            st.markdown("---")

            # GỘP HÓA ĐƠN THEO TÊN BÁNH VÀ TÍNH TỔNG TIỀN
            annotated_image = st.session_state.analyzed_image.copy()
            total_bill = 0
            total_items = 0
            bill_summary = {} # Dùng dictionary để gộp món trùng nhau

            # Quét danh sách detections để cộng dồn vào bill_summary và vẽ khung
            for i, det in enumerate(st.session_state.detections):
                cake_name = det['corrected_class']
                qty = det['quantity']

                # NẾU SỐ LƯỢNG LÀ 0 -> BỎ QUA, KHÔNG VẼ, KHÔNG TÍNH TIỀN
                if qty == 0:
                    continue

                # Gộp bánh vào hóa đơn
                if cake_name in bill_summary:
                    bill_summary[cake_name] += qty
                else:
                    bill_summary[cake_name] = qty

                # Vẽ khung
                x, y, x2, y2 = det['bbox']
                w, h = x2 - x, y2 - y
                current_menu = st.session_state.menu
                price_row = current_menu[current_menu["Tên Bánh (AI Nhận Diện)"] == cake_name]
                price = int(price_row["Giá Tiền (VNĐ)"].values[0]) if not price_row.empty else 0

                clean_name = remove_accents(cake_name)
                cv2.rectangle(annotated_image, (x, y), (x+w, y+h), (0, 255, 0), 3)

                # Nếu số lượng > 1 thì ghi rõ (x2, x3...) lên khung ảnh
                box_text = f"{clean_name} (x{qty})" if qty > 1 else f"{clean_name} ({price//1000}k)"
                org = (x, max(0, y - 10))
                cv2.putText(annotated_image, box_text, org, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 4)
                cv2.putText(annotated_image, box_text, org, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

            # XUẤT RA GIAO DIỆN
            col1, col2 = st.columns([2, 1])
            with col1:
                st.markdown("### 🖼️ Ảnh tổng quan")
                st.image(annotated_image, use_container_width=True)

            with col2:
                st.markdown("### 🧾 HÓA ĐƠN CHI TIẾT")
                st.markdown("---")

                # In các dòng hóa đơn đã được gộp
                for name, qty in bill_summary.items():
                    price_row = st.session_state.menu[st.session_state.menu["Tên Bánh (AI Nhận Diện)"] == name]
                    price = int(price_row["Giá Tiền (VNĐ)"].values[0]) if not price_row.empty else 0
                    subtotal = price * qty
                    total_bill += subtotal
                    total_items += qty
                    st.markdown(f"- **{name}:** {qty} x {price:,} = **{subtotal:,} VNĐ**")

                st.markdown("---")
                st.markdown(f"🎂 **TỔNG SỐ BÁNH:** {total_items}")
                st.markdown(f"💰 **TỔNG TIỀN: <span style='color:red; font-size:24px'>{total_bill:,} VNĐ</span>**", unsafe_allow_html=True)

                # MÃ QR THANH TOÁN
                st.markdown("---")
                st.markdown("### 📱 Quét mã để thanh toán")
                if os.path.exists(qr_code_path):
                    st.image(qr_code_path, caption="Mã QR Thanh Toán", use_container_width=True)
                    st.info("Vui lòng kiểm tra kỹ số tiền trước khi chuyển khoản.")
                else:
                    st.warning("⚠️ Chưa tìm thấy ảnh mã QR. Vui lòng tải file qr.png lên Google Colab (nằm ở /content/qr.png).")

Overwriting app.py


In [ ]:

import subprocess
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'])

# 3. Tải và cấu hình Cloudflare Tunnel
!wget -q -O cloudflared-linux-amd64 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("\n" + "="*50)
print("🚀 HỆ THỐNG ĐANG TẠO ĐƯỜNG TRUYỀN...")
print("Hãy tìm đường link có đuôi là: .trycloudflare.com ở ngay các dòng bên dưới nhé!")
print("="*50 + "\n")

# 4. Mở đường hầm
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501


🚀 HỆ THỐNG ĐANG TẠO ĐƯỜNG TRUYỀN...
Hãy tìm đường link có đuôi là: .trycloudflare.com ở ngay các dòng bên dưới nhé!

2026-06-21T01:51:34Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-21T01:51:34Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-21T01:51:40Z INF +--------------------------------------------------------------------------------------------+
2026-06-21T01:51:40Z INF |  Your quick Tunnel has been created! Visit it at (it may